In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Load environment variables from .env file
load_dotenv()

# Verify API keys are loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please create a .env file with your API key.")
if not os.getenv("PINECONE_API_KEY"):
    raise ValueError("PINECONE_API_KEY not found in environment variables. Please create a .env file with your API key.")

print("✅ API keys loaded successfully!")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ LLM and Embeddings initialized!")

/Users/carolinanami/Desktop/Ironhack/Week 1/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ API keys loaded successfully!
✅ LLM and Embeddings initialized!


In [2]:
# Query rewriting example
def rewrite_query(original_query):
    prompt = f"Rewrite this query to be more specific and better suited for document retrieval: {original_query}"
    response = llm.invoke(prompt)
    return response.content

# Test it out
original = "Tell me about AI"
rewritten = rewrite_query(original)
print(f"Original: {original}")
print(f"Rewritten: {rewritten}")

Original: Tell me about AI
Rewritten: Please provide an overview of artificial intelligence, including its key concepts, applications, and recent advancements in the field. Additionally, include information on ethical considerations and challenges associated with AI development.


In [3]:
# Sub-query decomposition - breaking complex questions into smaller ones
def decompose_query(complex_query):
    prompt = f"Break this complex query into 2-3 simpler sub-queries: {complex_query}"
    response = llm.invoke(prompt)
    return response.content

# Test it out
complex = "How does machine learning compare to deep learning and what are their applications?"
sub_queries = decompose_query(complex)
print(f"Complex query: {complex}")
print(f"\nSub-queries:\n{sub_queries}")

Complex query: How does machine learning compare to deep learning and what are their applications?

Sub-queries:
To break down the complex query into simpler sub-queries, we can focus on distinct aspects of the comparison and applications of machine learning and deep learning. Here are three sub-queries:

1. **What are the key differences between machine learning and deep learning?**
   
2. **What are the main applications of machine learning?**

3. **What are the main applications of deep learning?**

These sub-queries allow for a clearer exploration of the concepts and their respective uses.


In [4]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a parent document (like a long paragraph)
parent_docs = [
    Document(page_content="Machine learning is a subset of artificial intelligence. Deep learning uses neural networks with many layers. Natural language processing helps computers understand human language. Computer vision enables machines to see and identify objects. Reinforcement learning trains models through trial and error.")
]

print(f"Parent document length: {len(parent_docs[0].page_content)} characters")
print(f"Parent document: {parent_docs[0].page_content}")

Parent document length: 304 characters
Parent document: Machine learning is a subset of artificial intelligence. Deep learning uses neural networks with many layers. Natural language processing helps computers understand human language. Computer vision enables machines to see and identify objects. Reinforcement learning trains models through trial and error.


In [ ]:
# Split into small chunks (like cutting a long paragraph into smaller sentences)
small_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
small_chunks = small_splitter.split_documents(parent_docs)

# Add parent ID to each chunk (like labeling each piece with which article it came from)
for i, chunk in enumerate(small_chunks):
    chunk.metadata["parent_id"] = 0  # Reference to parent document

print(f"Created {len(small_chunks)} small chunks from 1 parent document")
print("\nAll chunks:")
for i, chunk in enumerate(small_chunks):
    print(f"Chunk {i+1}: {chunk.page_content}")